In [6]:
# Writes a list of elements in the number field of degree n as a matrix, where the ith column represents the 
# ith element with respect to the power basis 1, a, a^2, ..., a^(n-1). 

def elems_to_matrix_power (l, n):
    M = [x.list() + [0] * (n - len(x.list())) for x in l]
    return Matrix(M).transpose()

In [7]:
# Represents elements as a matrix where 
# each column is the corresponding element given in terms of the integral basis. 

def elems_to_basis (x, B):
    return (elems_to_matrix_power(B,len(B))).inverse() * (elems_to_matrix_power (x,len(B)))

In [8]:
# From a m x n - matrix of lists (of length r), constructs a r x (mn) matrix that has those lists as columns: 
# The entry (i,j) corresponds to the ni + j column 
def matrix_of_matrix_lists (M, m, n):
    L = []
    for i in range(m):
        for j in range(n):
            L = L + [M[i][j]]
    M = Matrix(L).transpose()
    return(M)

In [9]:
# Compute a Z-basis for the ideal I in upper-triangular form. The columns are the elements wrt the integral basis B.
def IntegralBasisIdeal (I, B):
    Z_basis = I.basis()
    W = (((Matrix(ZZ, n, n, (elems_to_basis(Z_basis,B))).transpose())[: , ::-1].echelon_form())[: , ::-1])[::-1 , :].transpose()
    return W 

In [10]:
# Given an 'almost' orthogonal basis given as columns of a matrix M, 'project' a vector b on the lattice spanned by the 
# basis and subtract it from the original vector. We pick the number of vectors from the basis that results in the minimal 
# euclidean length of the result. 

def PseudoProject (M, b): 
    n = len(M.columns())
    if n == 0 :
        return b , float(norm(b))
    else :
        L = []
        x = 0 * M[:, 0]
        S = []
        for i in range(n):
            L = L + [floor((vector(M[: , i]).dot_product(vector(b))) / (vector(M[: , i]).dot_product(vector(M[: , i]))))]
        for m in range(n):
            for j in range(m) : 
                x = x + L[j] * M[:, j]
            S = S + [(vector(b) - vector(x), norm(vector(b) - vector(x)))]
        Ls = sorted (S, key = lambda x: x[1])
    
        return Ls[0][0], float(Ls[0][1])

In [11]:
#Given a matrix A and vector b, solve the system A*x = b in the integers. We use the HNF of A for this.  
# After obtaining a result b', we approximately project it into the lattice of kernel elements of the matrix, and then 
# substract the projection from b in order to obtain a solution of smaller norm. 

def SolveRightInteger(M,b): 
    MAux, Echel = M.transpose().echelon_form(transformation=True)
    faux = Echel.transpose() * (MAux.transpose()).solve_right(b)
    faux, nm = PseudoProject(Echel.transpose()[: , len(M.rows()) : ], faux)
    return faux

In [12]:
# The generators of the ideal as an O-module are given in ideal_gens. 

def IdealEqSpanCertificate(K, ideal_gens,B): 
    O = K.ring_of_integers()
    I = O.ideal(ideal_gens)
    
    Z_basis = I.basis()
    W = (((Matrix(ZZ, n, n, (elems_to_basis(Z_basis, B))).transpose())[: , ::-1].echelon_form())[: , ::-1])[::-1 , :].transpose()
    M = Matrix(K, len(ideal_gens), n)
    MB = [[[] for j in range(n)] for i in range(len(ideal_gens))]

    for i in range(len(ideal_gens)):
        for j in range(n):
            M[i, j] = (ideal_gens[i]) * B[j]
            MB[i][j] = (elems_to_basis ([M[i,j]], B)).list()

    Mland = Matrix(ZZ, matrix_of_matrix_lists(MB,len(ideal_gens), len(B)))
    faux = [() for k in range(n)]
    # We solve the system Mland * x = W k in the integers. 
    faux = [() for k in range(n)]
    for k in range(n):
        faux[k] = SolveRightInteger(Mland, (vector(W[:, k])))

    f = [[[[] for j in range(n)] for i in range(len(ideal_gens))] for k in range(n)]
    for k in range(n):
        for i in range(len(ideal_gens)):
            for j in range(n):
                f[k][i][j] = faux[k][n * i + j]
               
    g = [[[[] for k in range(n)] for j in range(n)] for i in range(len(ideal_gens))]
    for i in range(len(ideal_gens)):
        for j in range(n) :
            g[i][j] = (W.solve_right(vector(MB[i][j]))).list()
    
    w = [[] for i in range(n)]  
    v = [[] for i in range(len(ideal_gens))]
    
    for i in range(len(ideal_gens)):
        v[i] = (elems_to_basis ([ideal_gens[i]], B)).list()
    for j in range(n):
        w[j] = (W[: , j]).list()
        
    return Mland, f, g, v, w

In [13]:
#Given a string, this outputs the string corresponding to placing a ! mark in front of every [ 

def ExList (string) : 
    l = ''
    for x in string:
        if x == '[' :
            l = l + '!['
        else : 
            l = l + x
    return l

In [14]:
# Compute multiplication table for an order in a number field by giving the defining polynomial f and a list with the basis elements. 
def MultiplicationTable(f, B):
    n = len(B)
    K.<a> = NumberField(f)
    #Columns of this matrix are basis for Ok in terms of 1, a^2, ..., a^n. It is upper triangular. 
    A = [[[] for j in range(n)] for i in range(n)]
    for i in range(n):
        for j in range(n):
            A[i][j] = (elems_to_basis([B[i] * B[j]], B)).list()
    return A

In [15]:
def IdealEqSpanCertificateLean(K, ideal_gens,B,name,ideal_name):
    Mland, f, g, v, w = IdealEqSpanCertificate(K, ideal_gens,B)
    print ("def " + name + ": IdealEqSpanCertificate O ℤ timesTableO " + ideal_name + '\n '+ ExList(str(v)) + ' \n ' + ExList(str(w)) + ' where')
    print("""  T := Table 
  heq := timesTableT_eq_Table
  hieq := rfl """)
    bt = ZZ(len((Mland[0 , :]).list()) / len(ideal_gens))
    M = [[[] for j in range(n)] for i in range(len(ideal_gens))]
    for i in range(len(ideal_gens)):
        for j in range(bt):
            M[i][j] = (Mland [: , bt * i + j]).list()
    print("  M :=" + ExList(str(M)))
    print("  hmulB := by decide")
    print("  f := " + ExList(str(f)))
    print("  g := " + ExList(str(g)))
    print("  hle1 := by decide")
    print("  hle2 := by decide")
    return w 

In [16]:
#Given the O-generators a_1, ..., a_k of an ideal in a number field K with basis B and an element in this ideal, the function 
# computes elements x1, x2, .., x_k in the ring of integers such that x1 * a_1 + ... + x_k * a_k = elem. 

def IdealLift(K, B, ideal_gens, elem): 
    n = len(B)
    O = K.ring_of_integers()
    
    I = O.ideal(ideal_gens)
    
    M = Matrix(K, len(ideal_gens), n)
    MB = [[[] for j in range(n)] for i in range(len(ideal_gens))]
    
    for i in range(len(ideal_gens)):
        for j in range(n):
            M[i, j] = (ideal_gens[i]) * B[j]
            MB[i][j] = (elems_to_basis ([M[i,j]], B)).list()
    
    Mland = Matrix(ZZ, matrix_of_matrix_lists(MB,len(ideal_gens), n))
    faux = SolveRightInteger(Mland, (vector(elems_to_basis([elem], B))))
    #print(faux)
    flag = 0 
    for z in faux:
        if z not in ZZ:
            flag = 1
            print('Not inside the ideal')
            return 0 
    k = len(ideal_gens)
    x = [0 for j in range(k)]
    
    for j in range(k):
        
        for i in range(n):
            x[j] = x[j] + faux[n * j + i]* B[i]

    check = 0 
    for j in range(k):
        check = check + x[j] * ideal_gens[j]
    if check == elem:
        return x 
    else:
        print('Something went wrong.')
        

In [17]:
def IdealMultiplication (K, B, ideal_gens1, ideal_gens2):
    O = K.ring_of_integers()
    I1 = O.ideal(ideal_gens1)
    I2 = O.ideal(ideal_gens2)
    ideal_gens3 = (I1 * I2).gens_reduced() 
    M = [[[] for j in range(len(ideal_gens2))] for i in range(len(ideal_gens1))]
    Mcoord = [[[] for j in range(len(ideal_gens2))] for i in range(len(ideal_gens1))]
    for i in range(len(ideal_gens1)):
        for j in range(len(ideal_gens2)):
            M[i][j] = ideal_gens1[i] * ideal_gens2[j]    
            Mcoord[i][j] = (elems_to_basis([M[i][j]], B)).list()
    Maux = []
    for i in range(len(ideal_gens1)):
        Maux = Maux + M[i]
    f = [[[[] for i in range(len(ideal_gens2))] for j in range(len(ideal_gens1))] for k in range(len(ideal_gens3))]
    for i in range(len(ideal_gens3)): 
        faux = IdealLift(K, B, Maux, ideal_gens3[i])
        for j in range(len(ideal_gens1)):
            for k in range(len(ideal_gens2)):
                f[i][j][k] = elems_to_basis([faux[(len(ideal_gens2)) * j + k]], B).list() 
    g = [[[[] for i in range(len(ideal_gens3))] for j in range(len(ideal_gens2))] for k in range(len(ideal_gens1))]
    for i in range(len(ideal_gens1)):
        for j in range(len(ideal_gens2)) : 
            for k in range(len(ideal_gens3)): 
                g[i][j][k] =  elems_to_basis([IdealLift(K, B, ideal_gens3, M[i][j])[k]], B).list()
    return ideal_gens3, Mcoord, f, g
            

In [18]:
def IdealMulEqCertificateLean (nameC, K, B, ideal_gens1, ideal_gens2, proof1_name, proof2_name, ideal1_name,ideal2_name) :
    ideal_gens3, Mcoord, f, g = IdealMultiplication(K,B,ideal_gens1, ideal_gens2)
    print(f"def {nameC} : IdealMulEqCertificate O ℤ timesTableO " + ideal1_name + ' ' + ideal2_name)
    ideal_gens1_coord = [ elems_to_basis([ideal_gens1[i]], B).list() for i in range(len(ideal_gens1))]
    ideal_gens2_coord = [ elems_to_basis([ideal_gens2[i]], B).list() for i in range(len(ideal_gens2))]
    ideal_gens3_coord = [ elems_to_basis([ideal_gens3[i]], B).list() for i in range(len(ideal_gens3))]
    print(' ', ExList(str(ideal_gens1_coord)), ExList(str(ideal_gens2_coord)))
    print(' ', ExList(str(ideal_gens3_coord)), 'where')
    print(f""" T := Table
 heq := timesTableT_eq_Table
 hI1 := {proof1_name}
 hI2 := {proof2_name}""")
    print(' M := ', ExList(str(Mcoord)))
    print(' hmul := by decide')
    print(' f := ', ExList(str(f)))
    print(' g := ', ExList(str(g)))
    print(' hle1 := by decide')
    print(' hle2 := by decide')
    return ideal_gens3
    

In [19]:
def IteratedMulLean (K, B, ideal_gens, ideal_pows, proof1_name, proof2_name, ideal_name): 
    if len(ideal_gens)> 1:
        ideal_name_accum = ideal_name + '0'
    else : 
        ideal_name_accum = ideal_name 
    m = len(ideal_gens)
    ideal_gens_accum = ideal_gens[0]
    cont = 0 
    proof_name = proof1_name
    for i in range(m):
        if i == 0:
            for j in range(ideal_pows[i] - 1) :
                
                if len(ideal_gens)> 1:
                    ideal_gens_accum = IdealMulEqCertificateLean('Mul' + str(ideal_name) + str(cont),K, B, ideal_gens_accum , ideal_gens[i], proof_name, proof2_name, '(' + ideal_name_accum + ')', str(ideal_name) + '0')
                    ideal_name_accum = ideal_name_accum + '*' + str(ideal_name) + '0'
                else : 
                    ideal_gens_accum = IdealMulEqCertificateLean('Mul' + str(ideal_name) + str(cont),K, B, ideal_gens_accum , ideal_gens[i], proof_name, proof2_name, '(' + ideal_name_accum + ')', str(ideal_name) )
                    ideal_name_accum = ideal_name_accum + '*' + str(ideal_name) 
                print('')
                proof_name = "ideal_eq_mul_of_IdealMulEqCertificate O ℤ timesTableO _ _ _ _ _ " + 'Mul' + str(ideal_name) + str(cont)
                cont = cont + 1
        else : 
            for j in range(ideal_pows[i]) :
                ideal_gens_accum = IdealMulEqCertificateLean('Mul' + str(ideal_name) + str(cont), K, B, ideal_gens_accum, ideal_gens[i], proof_name, proof2_name, '(' + ideal_name_accum + ')', str(ideal_name) + str(i))
                ideal_name_accum = ideal_name_accum + '*' + str(ideal_name) + str(i)   
                print('')
                proof_name = "ideal_eq_mul_of_IdealMulEqCertificate O ℤ timesTableO _ _ _ _ _ " + 'Mul' + str(ideal_name) + str(cont)
                cont = cont + 1
    return ideal_gens_accum 

In [20]:
# Certificate of membership using a Z-basis. Input is the generators of the ideal as an O-module and the coordinates of the 
# element. 
def IdealMemZLean (K, B,ideal_gens, mem_coords, name_lemma, name_ideal, cert_name_eq) :
    Mland, f, g, v, w = IdealEqSpanCertificate(K, ideal_gens,B)
    g = Matrix(w).transpose().solve_right(vector(mem_coords))
    flag = 0
    for x in g:
        if x not in ZZ:
            print('Element is not a member of the ideal')
            flag = 1
    if flag == 0 :
        print(f"def {name_lemma} : IdealMemCertificate O ℤ B {name_ideal}")
        print(' ',ExList(str(w)), ExList(str(mem_coords)), 'where')
        print(f' hieq := ideal_eq_of_IdealEqSpanCertificate _ _ {cert_name_eq}')
        print(' g :=', ExList(str(g.list())))
        print(' hmem := by decide')

In [21]:
def IsOrderOf (p, name): 
    print (f'def {name} : IsOrderOf ({primitive_root(p)} : ZMod {p}) {p-1} where')
    F = factor(p-1)
    m = len(F)
    print(' m :=', m)
    P = [x[0] for x in F]
    e = [x[1] for x in F]
    print(' P :=', ExList(str(P)))
    print(' e :=', ExList(str(e)))
    print(""" hP := by decide
 hm := by rfl
 hid := by decide
 hnid := by decide""")

In [22]:
# We are brute forcing the computation of this discrete logarithm. If we have a map to Z/nZ, we can use more efficient 
# methods. Find log(x) with basis z. 

def DiscreteLog (O, I, x, z): 
    R = QuotientRing(O , I, 'y')
    for i in range(I.norm()):
        if R(x) == R(z) ^ i :
            return i , Mod(z ^ i, I.norm())

This part is to compute the ideals and the submatrix that makes everything work. 

In [91]:
# Output: 
# The matrix proving that x is not a p-power times a unit. Which proves that if J ^ p = (x) implies that J is not principal. 
# The list of prime ideals of degree 1 that correspond to the rows of the matrix. 
# The list of elements corresponding to the columns of the matrix: generators for the unit group mod torsion, the torsion unit (if p divides 
# the torsion order) and x. 
# A flag indicating if p divides torsion order (flag = 1) or not (flag = 0).

#The order of the ideals is in such a way that the submatrix obtained by eliminating the last column and the last row is of full-rank. 

def MatrixPrimes (K, O, p, x, T) :
    # L is the list of elements representing the columns. 
    L = []
    U = K.unit_group().gens()

    flag_dvdtor = 0 
    
    for i in range(len(U)):
        if i == 0 :
            if order(K.unit_group().gens()[0]) % p == 0: # If p divides torsion order, we include it in the columns 
                L = L + [K.unit_group().gens()[0]]
                flag_dvdtor = 1
        else : 
            L = L + [K.unit_group().gens()[i]]
            
    if flag_dvdtor == 1 : 
        # Move the first column to the last 
        L = L[1:] + [L[0]]+ [x]
    else : 
        L = L + [x]
    
    Ql = []
    A = []
    cont = 0 
    for q in primes(T + 1):
        if (q - 1) % p == 0 : 
            z = primitive_root(q)
            F = factor(O.ideal(q))
            for Q in F:
                if Q[0].norm() == q :
                    B = A + [[DiscreteLog(O, Q[0], L[j], z)[0] % p for j in range(len(L))]]
                    if Matrix(B).change_ring(GF(p)).rank() > cont :
                        Ql = Ql + [Q]
                        cont = cont + 1
                        A = B 
        if cont == len(L) :
            pivots = Matrix(GF(p), A)[:, :-1].transpose().pivots()
            A_initial = [A[i] for i in pivots]
            A_last = [A[i] for i in range(len(L)) if i not in pivots]
            Ql_initial = [Ql[i][0] for i in pivots]
            Ql_last = [Ql[i][0] for i in range(len(L)) if i not in pivots]
            A = A_initial + A_last
            Ql = Ql_initial + Ql_last
            
            return Matrix(A), Ql, L, flag_dvdtor
            
    return 0 

In [24]:
# Note that this assumes the prime is of inertia degree one. 

def DiscreteLogCLean (K, B, name, ideal_gens, hcard, hprim_root, p, x, x_name, name_ideal, cert_name_eq):
    mem_coords = elems_to_basis([x], B).list()
    O = K.ring_of_integers()
    I = O.ideal(ideal_gens)
    q = I.norm()
    k, m = DiscreteLog(O, I, x, primitive_root(q))
    mem_coords[0] = mem_coords[0] - ZZ(m)
    IdealMemZLean(K, B, ideal_gens, mem_coords, name + 'Mem', name_ideal, cert_name_eq)
    print('')
    print(f'def {name}: DiscreteLogCertificate {hcard} {hprim_root} {p} {x_name} {k % p} where')
    print(' r :=', len(B))
    print(' hN := by infer_instance')
    print(' hpdvd := by decide')
    print(' B := B')
    print(' hone := B_one')
    print(' xcoord := ', ExList(str(elems_to_basis([x], B).list())))
    print(' hxeq :=  rfl')
    print(' m:=', m)
    print(' C :=', ExList(str(mem_coords)))
    print(' hCeq := by rfl')
    print(' hmem := mem_of_certificate O ℤ _ _ _ _ ' + name + 'Mem' )
    print(' k := ', k)
    print(' hpow := by decide')
    print(' heql := by decide')
    return k % p , m

In [25]:
# Given a list of prime ideals of degree 1, and a list of elements. Prove primitive roots and compute 
# certificates for the discrete log of those elements in O/I. 

#IdealEqSpanCertificateLean(K, ideal_gens,B,name,ideal_name):

def FindNzEntry (w) : 
    for i in range(len(w)):
        for j in range(len(w[0])):
            if w[i][j] != 0:
                return i, j

def DiscreteLogListLean  (K, B, list_ideal_gens, ideal_name, p, list_elems, name_elems):
    O = K.ring_of_integers()
    N = set([O.ideal(x).norm() for x in list_ideal_gens])
    L = zero_matrix(len(list_ideal_gens), len(list_elems)) # The matrix of logarithms
    M = zero_matrix(len(list_ideal_gens), len(list_elems)) # The matrix of values under the isomorphism O/I \to mod q. 
    for q in N: 
        if q < 100:
            print(f'instance hq{q} : Fact $ Nat.Prime {q} := by decide')
        else : 
            print(f'instance hq{q} : Fact $ Nat.Prime {q} := by sorry') #In case of a big prime, we sorry the proof for now. 

    print('') 
    for q in N: 
        IsOrderOf(q, 'R' + str(q))
        print('')

    for i in range(len(list_ideal_gens)): 
        gensc = [elems_to_basis([x], B).list() for x in list_ideal_gens[i]]
        print(f'def {ideal_name}{i} : Ideal O := Ideal.span (Set.range (fun i ↦ B.equivFun.symm (' + ExList(str(gensc)) + ' i)))')
    print('')
        
    for i in range(len(list_ideal_gens)): 
        w = IdealEqSpanCertificateLean(K, list_ideal_gens[i], B, 'A' + str(i), ideal_name + str(i))
        ik, jk = FindNzEntry(w) 
        print('')
        print(f'lemma N{i} : Nat.card (O ⧸ {ideal_name}{i}) = {O.ideal(list_ideal_gens[i]).norm()} := ')
        print(f" ideal_norm_eq_prod' B _ _ (by decide) {ik} {jk} (by decide) (ideal_eq_of_IdealEqSpanCertificate O ℤ A{i})")            
        print('')

    for i in range(len(list_elems)): 
        print(f'def {name_elems[i]} := B.equivFun.symm', ExList(str(elems_to_basis([list_elems[i]], B).list())))
        print('')

    for i in range(len(list_ideal_gens)):
        for j in range(len(list_elems)): 
            l, m = DiscreteLogCLean(K, B, 'Log' + str(i) + str(j), list_ideal_gens[i], 'N' + str(i), f'((orderOf_of_IsOrderOf R{O.ideal(list_ideal_gens[i]).norm()}) ▸ IsPrimitiveRoot.orderOf _)', p, list_elems[j], name_elems[j], ideal_name + str(i), 'A' + str(i))
            print('')
            L[i,j] = l 
            M[i,j] = m 
    return L, M  

In [26]:
# Write a proof that I ^ m = <mem_elem>. 

def IdealPowLean (B, lemma_name , ideal_gens, ideal_name, elem, elem_name, m): 
    gensc = [elems_to_basis([x], B).list() for x in ideal_gens]
    print(f'def {ideal_name} : Ideal O := Ideal.span (Set.range (fun i ↦ B.equivFun.symm (' + ExList(str(gensc)) + ' i)))')
    a = IteratedMulLean(K, B, [ideal_gens], [m], 'rfl', 'rfl' , ideal_name)
    if a[0] != elem : 
        print("The generators don't match. Equal generators must be given")
        print('Computed: ', a[0])
        print('Given: ', elem)
    else: 
        print(f"lemma {lemma_name} : {ideal_name} ^ {m} = ", "Ideal.span {" + elem_name + "} := by")
        print(f' simp only [pow_succ, pow_one, pow_zero, one_mul]')
        print(f' simp [ideal_eq_mul_of_IdealMulEqCertificate O ℤ timesTableO _ _ _ _ _ Mul{ideal_name}{m - 2}, {elem_name}]')
        print(f' rfl')
    

In [27]:
# Produce a prime ideal that proves that p does not divide the torsion. 
def PrimePNeDvd (K, B, p): 
    O = K.ring_of_integers()
    for q in prime_range(1,100): 
        if (q - 1) % p != 0 :
            F = factor(O.ideal(q))
            for I in F:
                if I[0].norm() == q: 
                    return I[0], [x for x in I[0].gens_reduced()]
                    
#[elems_to_basis([I[0].gens_reduced()[i]], B).list() for i in range(len(I[0].gens_reduced()))]

In [28]:
# Produces a proof that p does not divide the torsion order. If the degree of the number field is odd, then it does so 
# with the proof 'finrank_proof', otherwise it looks for a prime ideal of norm prime q such that p does not divide q-1. 
def pNeDvdTorsionLean(K, B, finrank_proof, integral_closure_eq, ideal_name, p): 
    if K.degree() % 2 != 0 and finrank_proof != '' :
        print(f'lemma ne_dvd_torsion{p} : ¬{p} ∣ Nat.card ↥(CommGroup.torsion (↥O)ˣ) := by')
        print(f""" suffices ¬ {p} ∣ 2 from ?_
 · convert this 
   rw [{integral_closure_eq}, ← Fintype.card_eq_nat_card,
   ←  NumberField.Units.torsionOrder_eq_two_of_odd_finrank (by rw [{finrank_proof}] ; decide)]
   rfl
 · decide""")
    else : 
        I, gens = PrimePNeDvd (K, B, p)
        gensc = [elems_to_basis([gens[i]], B).list() for i in range(len(gens))]
        print(f'def {ideal_name} : Ideal O := Ideal.span (Set.range (fun i ↦ B.equivFun.symm (' + ExList(str(gensc)) + ' i)))')
        print('')
        IdealEqSpanCertificateLean(K, gens, B, f'A{ideal_name}', ideal_name)
        print('')
        print(f' lemma ne_dvd_torsion{p} : ¬{p} ∣ Nat.card ↥(CommGroup.torsion (↥O)ˣ) := by ')
        print(f"""  refine prime_not_dvd_torsion_of_not_dvd (by decide) {ideal_name} 
     (ideal_norm_eq_prod' B _ _ (by decide) 0 0 (by decide) (ideal_eq_of_IdealEqSpanCertificate O ℤ A{ideal_name}))
     (by decide) (by decide)""")
        

In [29]:
#Generate proofs that units (defeq to B.equivFun.symm ![]) are indeed units, by computing and multiplying with the 
# inverse. An alternative could be computing the norm of I/(zeta) to be equal to 1. But this requires n multiplications. 

def IsUnitInvLean (B, list_elems, list_names):
    list_elems_inv = [x ^ (-1) for x in list_elems]
    gensc = [elems_to_basis([x], B).list() for x in list_elems_inv]
    for i in range(len(list_elems)): 
        print(f'lemma isUnit_{list_names[i]} : IsUnit {list_names[i]} := by ')
        print(f' apply isUnit_of_mul_eq_one _ (B.equivFun.symm {ExList(str(gensc[i]))})')
        print(""" rw [← B_one_repr]
 refine table_mul_list_eq_mul timesTableO.table B _ _ _ timesTableO.basis_mul_basis ?_
 rw [← table_mul_eq_table_mul' _ _ timesTableT_eq_Table]
 decide""")
        print('')

In [1]:
# Computes a pseudo-remainder sequence, using the constants in the sub-resultant algorithm that avoid coefficient explotion (see
# algorithm 4.1.11 in Cohen). 
# Alternatively, one can define f by computing the content of the polynomials, but this results in larger coefficients. 

def SturmSequencePseudo (p, q): 
    R.<X> = PolynomialRing(ZZ)
    P = [p, q]
    if gcd(p,q) != 1:
        print('Polynomials p and q are not coprime') 
    else: 
        e = []
        f = []
        Q = []
        deg = p.degree()
        i = 0 
        ### -- aux Sub-resultant constants 
        g = 1
        h = 1
        ##
        while deg > 0 and i < 100: 
            delta = P[i].degree() - P[i + 1].degree()
            e = e + [abs(((P[i + 1]).list()[-1]) ^ (delta + 1))]
            paux = e[i] * P[i] 
            p_i2 = - paux % P[i + 1]
            Q = Q + [ZZ[X]((paux + p_i2) / P[i + 1])]
            f = f + [g * (h ^ delta)]
            #f = f + [gcd(p_i2.list())]
            P = P + [p_i2 / f[i]]
            deg = p_i2.degree()
            i = i + 1
            ## -- aux Sub-resultant constants
            g = abs((P[i]).list()[-1])
            h = h ^ (1 - delta) * g ^ delta
            ## 
        return e, f, [q.list() for q in Q], [p.list() for p in P]


In [4]:
#Takes a separable polynomial with integer coefficients, produces a Lean certificate to count the real roots. 

def SturmCertificateLean (p, name): 
    q = p.derivative()
    e, f, Q, P = SturmSequencePseudo(p,q)
    print(f"""def {name} : SturmBuilderOfList {P} {P[0]} {P[1]} where
  hlen := by decide
  h0 := by decide
  h1 := by decide
  hlast := by decide
  hdrop := by decide
  hmono := by
    dsimp
    intro i hic 
    have hi : i < {len(P) - 1} := by omega
    interval_cases i <;> (dsimp ; decide)
  e := {e}
  f := {f}
  epos := by decide
  fpos := by decide
  Q := {Q}
  hel := by decide
  hfl := by decide
  hQl := by decide
  hrem := by
    dsimp
    intro i hi
    have hi : i < {len(P) - 2} := by omega
    interval_cases i <;> (dsimp ; decide)""")
    return P
    

In [32]:
def RankUnitCertificateLean (f, proof_ofList, proof_AdjoinRoot, proof_integral_closure_eq, name_cert): 
    R.<X> = PolynomialRing(ZZ)
    F.<x> = PolynomialRing(RR)
    P = SturmCertificateLean(f, f'Sturm{name_cert}')
    k = len((F(f)).roots())
    r = (k + f.degree()) / 2 
    print('')
    print(f'def {name_cert} : RankUnitsCertificate O where')
    print(f'  f := {R(f)}')
    print(f"""  l := {R(f).list()}
  hl := {proof_ofList}
  hlz := by decide
  hz := by decide
  hAdj := {proof_AdjoinRoot}
  heq := {proof_integral_closure_eq}
  P := {P}
  SB := Sturm{name_cert}
  k := {k}
  r := {r}
  hr := by decide
  hreq := by decide """)

In [42]:
#With v a torsion unit. Write a proof that v ^ m = 1, where m is the order of v. 

def TorsionUnitProof (K, B, v, name): 
    m = K(v).multiplicative_order()
    print(f"""lemma v_pow_one : {name} ^ {m} = 1 := by
  rw [← B_one_repr]
  apply table_nPow_sq_table_eq_pow timesTableO.table Table B _ (timesTableO.basis_mul_basis) 
  timesTableT_eq_Table _ (by norm_num)
  decide""")
    return m

In [34]:
# primitive_root(p)
def CertificateNonPrincipalityNT (M, p, m, zeta_names, prime_ideal_name, ideal_name, elem_name, primes_q, name_rank_cert, ideal_pow_proof):
    u0 = ''
    for x in zeta_names : 
        if u0 == '':
            u0 = x
        else : 
            u0 = u0 + ', ' + x
    In = ''
    #print(list(M[0,:]))
    d = len(list(M[0,:])[0])
    for i in range(d):
        if In == '':
            In = prime_ideal_name+str(i)
        else : 
            In = In + ', ' + prime_ideal_name+str(i)
    M = Matrix(GF(p), M)
    print(f'def NPC{ideal_name} : NonPrincipalCertificateNDvdT {ideal_name} where ')
    print(f""" p := {p}
 hp := by decide
 r := {len(zeta_names)}
 huc := by 
  rw [units_finrank_of_RankUnitsCertificate {name_rank_cert}]
  decide
 u := ![{u0}]
 hu := fun i => 
  match i with """)
    for i in range(len(zeta_names)): 
        print(f'  | {i} => isUnit_{zeta_names[i]}')
    print(f' q := {ExList(str(primes_q))}')
    print(f""" hqP := by 
  intro i 
  match i with """)
    for i in range(len(primes_q)): 
        print(f'  | {i} => exact hq{primes_q[i]}')
    print(f' I := ![{In}]')
    print(f""" hcard := fun i =>
  match i with  """)
    for i in range(d): 
        print(f'  | {i} => N{i}')
    print(f' ζ := !{[primitive_root(q) for q in primes_q]}')
    print(f""" hr := fun i =>
  match i with """)
    for i in range(len(primes_q)): 
        print(f'  | {i} => ((orderOf_of_IsOrderOf R{primes_q[i]}) ▸ IsPrimitiveRoot.orderOf _)')
    print(f""" hdvd := by decide
 a := {elem_name}
 n := {m}
 hpdvd := by decide
 hJ := {ideal_pow_proof}
 hpndvdt := ne_dvd_torsion{p}
 M := {ExList(str([list(list(M[i,:])[0]) for i in range(d)]))}
 hM1 := by 
  intro j i hj 
  fin_cases i <;> interval_cases j """)
    for i in range(d): 
        for j in range(d - 1): 
            print(f'  · erw [eq_of_DiscreteLogCertificate Log{i}{j}] ; decide')
    print(f""" hM2 := by 
  intro i 
  match i with """)
    for i in range(d):
        print(f'  | {i} => exact Log{i}{d-1}')
    print(f""" hDl := by decide
 Minv := {ExList(str([list(list((M ^ (-1))[i,:])[0]) for i in range(d)]))}
 hInv := by decide
 N := {ExList(str([list(list((M[0 : d - 1, 0 : d - 1] ^ (-1))[i,:])[0]) for i in range(d - 1)]))}
 hNiv := by decide""")
    

In [48]:
def CertificateNonPrincipalityDVDT (M, p, m, zeta_names, v_name, order_v, prime_ideal_name, ideal_name, elem_name, primes_q, name_rank_cert, ideal_pow_proof):
    u0 = ''
    for x in zeta_names : 
        if u0 == '':
            u0 = x
        else : 
            u0 = u0 + ', ' + x
    In = ''
    #print(list(M[0,:]))
    d = len(list(M[0,:])[0])
    for i in range(d):
        if In == '':
            In = prime_ideal_name+str(i)
        else : 
            In = In + ', ' + prime_ideal_name+str(i)
    M = Matrix(GF(p), M)
    print(f'def NPC{ideal_name} : NonPrincipalCertificateDvdT {ideal_name} where ')
    print(f""" p := {p}
 hp := by decide
 r := {len(zeta_names)}
 huc := by 
  rw [units_finrank_of_RankUnitsCertificate {name_rank_cert}]
  decide
 u := ![{u0}]
 hu := fun i => 
  match i with """)
    for i in range(len(zeta_names)): 
        print(f'  | {i} => isUnit_{zeta_names[i]}')
    print(f' v := {v_name}')
    print(f' m := {order_v}')
    print(f' hm := by norm_num')
    print(f' hmv := v_pow_one')
    print(f' q := {ExList(str(primes_q))}')
    print(f""" hqP := by 
  intro i 
  match i with """)
    for i in range(len(primes_q)): 
        print(f'  | {i} => exact hq{primes_q[i]}')
    print(f' I := ![{In}]')
    print(f""" hcard := fun i =>
  match i with  """)
    for i in range(d): 
        print(f'  | {i} => N{i}')
    print(f' ζ := !{[primitive_root(q) for q in primes_q]}')
    print(f""" hr := fun i =>
  match i with """)
    for i in range(len(primes_q)): 
        print(f'  | {i} => ((orderOf_of_IsOrderOf R{primes_q[i]}) ▸ IsPrimitiveRoot.orderOf _)')
    print(f""" hdvd := by decide
 a := {elem_name}
 n := {m}
 hpdvd := by decide
 hJ := {ideal_pow_proof}
 M := {ExList(str([list(list(M[i,:])[0]) for i in range(d)]))}
 hM1 := by 
  intro j i hj 
  fin_cases i <;> interval_cases j """)
    for i in range(d): 
        for j in range(d - 1-1): 
            print(f'  · erw [eq_of_DiscreteLogCertificate Log{i}{j}] ; decide')
    print(f""" hM2 := by 
  intro j  
  fin_cases j """)
    for i in range(d): 
        print(f'  · erw [eq_of_DiscreteLogCertificate Log{i}{d-1-1}] ; decide')
    print(f""" hM3 := by 
  intro i 
  match i with """)
    for i in range(d):
        print(f'  | {i} => exact Log{i}{d-1}')
    print(f""" hDl := by decide
 Minv := {ExList(str([list(list((M ^ (-1))[i,:])[0]) for i in range(d)]))}
 hInv := by decide
 N := {ExList(str([list(list((M[0 : d - 1, 0 : d - 1] ^ (-1))[i,:])[0]) for i in range(d - 1)]))}
 hNiv := by decide""")